# CEG-WM Content-Curve LF/HF frequency response

Run all cells after setting Colab Secrets `CEG_WM_ROOT_KEY` and `HF_TOKEN`. This is standalone descriptive HF/LF frequency response only: the frozen plan has 8 units, 10 conditions, 4 arms, 40 records per unit, 320 total records, and 16 external wrong keys. Active resumable state remains local to Colab; Drive receives only complete checkpoint or final ZIP/checksum pairs. This notebook makes no winner, complementarity, joint-advantage, robustness, calibration/FPR, Content-gating, or scientific self-adjudication claim. Return the Drive pair for external supervisor authentication.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
import os
def _required_secret(name):
    value = os.environ.pop(name, None)
    if value is None:
        value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f'missing required Colab Secret: {name}')
    return value
root_key = _required_secret('CEG_WM_ROOT_KEY')
hf_token = _required_secret('HF_TOKEN')
run_store_root = Path('/content/drive/MyDrive/CEG-WM/stage_a_standalone_lf_hf_frequency_response')
run_store_root.mkdir(parents=True, exist_ok=True)


In [ ]:
import json, re, subprocess, sys
repo = Path('/content/CEG-WM-Content-Curve-exact')
if repo.exists():
    raise RuntimeError('detached checkout path already exists')
subprocess.run(['git', 'init', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'remote', 'add', 'origin', 'https://github.com/RICHAAARC/CEG-WM.git'], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '64', 'origin', 'refs/heads/Content-Curve-Evidence'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'b1a806a34a16435c4242e45eafa3818b3a37b8a6'], check=True)
resolved_exact = subprocess.run(['git', '-C', str(repo), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if re.fullmatch(r'[0-9a-f]{40}', resolved_exact) is None:
    raise RuntimeError('resolved Curve branch head is not an exact revision')
if subprocess.run(['git', '-C', str(repo), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout:
    raise RuntimeError('execution checkout is not clean')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)], check=True)


In [ ]:
local_work_root = Path('/content/cegwm-content-curve-local')
runner_env = dict(os.environ)
runner_env['CEG_WM_ROOT_KEY'] = root_key
runner_env['HF_TOKEN'] = hf_token
command = [sys.executable, '-m', 'experiments.stage_a_frequency_response.run_colab', '--repo-root', str(repo), '--expected-exact', resolved_exact, '--local-work-root', str(local_work_root), '--artifact-sink', str(run_store_root)]
run_id = None
last_committed = -1
terminal_event = None
try:
    process = subprocess.Popen(command, cwd=str(repo), env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    for line in process.stdout:
        if terminal_event is not None:
            raise RuntimeError('runner emitted stdout after terminal summary')
        if line.startswith('CEGWM_PROGRESS '):
            progress = json.loads(line.removeprefix('CEGWM_PROGRESS '))
            if set(progress) != {'run_id', 'committed', 'fixed_total', 'phase'}:
                raise RuntimeError('runner progress schema differs')
            candidate_run_id = progress['run_id']
            committed = progress['committed']
            if re.fullmatch(r'slhfr-[0-9a-f]{24}', candidate_run_id or '') is None:
                raise RuntimeError('runner progress has invalid deterministic run identity')
            if run_id is not None and run_id != candidate_run_id:
                raise RuntimeError('runner changed deterministic run identity')
            if type(committed) is not int or not 0 <= committed <= 8 or progress['fixed_total'] != 8:
                raise RuntimeError('runner progress count differs')
            if progress['phase'] not in {'identity_ready', 'resume_ready', 'unit_committed', 'checkpoint_published'}:
                raise RuntimeError('runner progress phase differs')
            if committed < last_committed:
                raise RuntimeError('runner progress committed count regressed')
            run_id, last_committed = candidate_run_id, committed
            print({'run_id': run_id, 'committed': committed, 'fixed_total': 8, 'phase': progress['phase']})
        elif line.startswith('CEGWM_SUMMARY '):
            terminal_event = json.loads(line.removeprefix('CEGWM_SUMMARY '))
            if set(terminal_event) != {'run_id', 'committed', 'fixed_total', 'phase', 'rc'}:
                raise RuntimeError('runner summary schema differs')
            if terminal_event['run_id'] != run_id or terminal_event['committed'] != 8 or terminal_event['fixed_total'] != 8 or terminal_event['phase'] != 'terminal' or type(terminal_event['rc']) is not int:
                raise RuntimeError('runner summary differs')
        elif line.strip():
            raise RuntimeError('runner emitted unrecognized stdout')
    runner_rc = process.wait()
finally:
    runner_env.pop('CEG_WM_ROOT_KEY', None)
    runner_env.pop('HF_TOKEN', None)
    root_key = hf_token = ''
    del root_key, hf_token, runner_env
if run_id is None:
    raise RuntimeError('runner produced no deterministic run identity')
if terminal_event is not None and terminal_event['rc'] != runner_rc:
    raise RuntimeError('runner summary return code differs')
if runner_rc == 0 and terminal_event is None:
    raise RuntimeError('successful runner has no terminal summary')


In [ ]:
drive_run_dir = run_store_root / run_id
zip_path = drive_run_dir / f'{run_id}.zip'
checksum_path = drive_run_dir / f'{run_id}.zip.sha256'
pair_present = zip_path.is_file() and checksum_path.is_file()
summary = {'run_id': run_id, 'resolved_exact': resolved_exact, 'runner_rc': runner_rc, 'zip_path': str(zip_path), 'checksum_path': str(checksum_path), 'pair_present': pair_present}
print(summary)
if not pair_present:
    raise RuntimeError('runner did not leave a complete terminal package pair for external validation')
if runner_rc != 0:
    raise RuntimeError('runner completed with a nonzero return code')
